In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os, random
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adam

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

BASE = '/content/drive/MyDrive/EdgeTB_Project/datasets'

# Same generators used but shuffle the labels after loading
datagen = ImageDataGenerator(rescale=1./255)
train_gen = datagen.flow_from_directory(
    f'{BASE}/train', target_size=(224, 224), batch_size=32,
    class_mode='binary', shuffle=False)   # shuffle=False so order is fixed

# Collect one pass of images and labels, then shuffle labels only
x_list, y_list = [], []
for i in range(len(train_gen)):
    x, y = train_gen[i]
    x_list.append(x); y_list.append(y)
    if i >= len(train_gen) - 1:
        break
x_all = np.concatenate(x_list)
y_all = np.concatenate(y_list)
y_shuf = y_all.copy()
np.random.shuffle(y_shuf)

print('label agreement after shuffle:', float(np.mean(y_all == y_shuf)))  # should be ~0.5

# Small model, few epochs
base = MobileNetV2(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base.trainable = False
z = GlobalAveragePooling2D()(base.output)
z = Dropout(0.3)(z)
out = Dense(1, activation='sigmoid')(z)
m = Model(base.input, out)
m.compile(optimizer=Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])

hist = m.fit(x_all, y_shuf, epochs=5, batch_size=32, validation_split=0.15, verbose=1)
print('final train acc:', round(hist.history['accuracy'][-1], 4))
print('final val acc:  ', round(hist.history['val_accuracy'][-1], 4))

Found 2940 images belonging to 2 classes.
label agreement after shuffle: 0.7170068027210884
9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 141s 2s/step - accuracy: 0.8111 - loss: 0.5189 - val_accuracy: 0.8549 - val_loss: 0.4373
Epoch 2/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 137s 2s/step - accuracy: 0.8187 - loss: 0.4891 - val_accuracy: 0.8549 - val_loss: 0.4286
Epoch 3/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 143s 2s/step - accuracy: 0.8215 - loss: 0.4871 - val_accuracy: 0.8549 - val_loss: 0.4258
Epoch 4/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 166s 2s/step - accuracy: 0.8275 - loss: 0.4764 - val_accuracy: 0.8549 - val_loss: 0.4290
Epoch 5/5
79/79 ━━━━━━━━━━━━━━━━━━━━ 140s 2s/step - accuracy: 0.8251 - loss: 0.4648 - val_accuracy: 0.8549 - val_loss: 0.4278
final train acc: 0.8251
final val acc:   0.8549


In [ ]:
from sklearn.metrics import roc_auc_score, accuracy_score

# predictions on the same data used above

probs = m.predict(x_all, batch_size=32).flatten()
pred  = (probs > 0.5).astype(int)

print('majority baseline (fraction of label 0):', round(float(np.mean(y_shuf == 0)), 4))
print('accuracy on shuffled labels:', round(accuracy_score(y_shuf, pred), 4))
print('AUC on shuffled labels:     ', round(roc_auc_score(y_shuf, probs), 4))
print('predicted positive rate:    ', round(float(pred.mean()), 4))
print('true positive rate in y_shuf:', round(float(y_shuf.mean()), 4))

92/92 ━━━━━━━━━━━━━━━━━━━━ 136s 1s/step
majority baseline (fraction of label 0): 0.8333
accuracy on shuffled labels: 0.8333
AUC on shuffled labels:      0.6435
predicted positive rate:     0.0
true positive rate in y_shuf: 0.1667
